In [1]:
%cd /kaggle/working

!git clone https://github.com/phusroyal/ViHOS.git
%cd /kaggle/working/ViHOS

!find . -maxdepth 3 -type f | sed 's#^\./##' | sort | head -100

/kaggle/working
Cloning into 'ViHOS'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 152 (delta 65), reused 102 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 5.80 MiB | 16.22 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/kaggle/working/ViHOS
codes/Code to tokenize data/Tokenize by Word.ipynb
codes/Code to tokenize data/Tokenizer by Syllabel.ipynb
codes/Models/BiLSTM-CRF _HOS_.ipynb
codes/Models/PhoBERT_HOS.ipynb
codes/Models/Where to download word2vec models?
codes/Models/XLMR_HOS.ipynb
codes/Requirement.txt
data/README.md
data/Span_Extraction_based_version/dev.csv
data/Span_Extraction_based_version/train.csv
data/Test_data/test.csv
.git/config
.git/description
.git/HEAD
.git/hooks/applypatch-msg.sample
.git/hooks/commit-msg.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/post-update.sample
.git/hooks/pre-applypatch.sample
.git/hooks/pre-commi

In [2]:
from sklearn.metrics import f1_score
import numpy as np
import os
from tqdm import tqdm

In [3]:
import tensorflow as tf

tf.keras.backend.clear_session()

# clear gpu memory using torch
import torch
torch.cuda.empty_cache()

# clear output
from IPython.display import clear_output
clear_output()

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
REPO_DIR = "/kaggle/working/ViHOS"
DATA_DIR = f"{REPO_DIR}/data"

train_path = f"{DATA_DIR}/Sequence_labeling_based_version/Syllable/train_BIO_syllable.csv"
dev_path   = f"{DATA_DIR}/Sequence_labeling_based_version/Syllable/dev_BIO_syllable.csv"

test_bio_path = f"{DATA_DIR}/Sequence_labeling_based_version/Syllable/test_BIO_syllable.csv"

test_gold_path = f"{DATA_DIR}/Test_data/test.csv"

test_path = test_bio_path

In [6]:
from transformers import (
    AutoModel, AutoConfig, XLMRobertaModel,
    AutoTokenizer, AutoModelForSequenceClassification
)

input_model = XLMRobertaModel.from_pretrained("xlm-roberta-large")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")
input_model.resize_token_embeddings(len(tokenizer))

clear_output()

# Data

In [7]:
import pandas as pd
import transformers
import torch
import torch.nn as nn
import pandas as pd

from IPython.display import clear_output
clear_output()

In [8]:
def prepare_data(file_path):
    df = pd.read_csv(file_path)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df.columns = [c.strip().lower() for c in df.columns]

    required_cols = {"word", "tag"}
    if not required_cols.issubset(set(df.columns)):
        raise ValueError(
            f"File {file_path} thiếu cột word/tag. "
            f"Cột hiện có: {df.columns.tolist()}"
        )

    sentences = []
    labels = []

    if "sentence_id" in df.columns:
        for _, group in df.groupby("sentence_id", sort=False):
            words = group["word"].astype(str).tolist()
            tags = group["tag"].astype(str).tolist()
            sentences.append(words)
            labels.append(tags)
    else:
        if "index" not in df.columns:
            raise ValueError(
                f"File {file_path} không có sentence_id hoặc index. "
                f"Cột hiện có: {df.columns.tolist()}"
            )

        cur_words = []
        cur_tags = []

        for _, row in df.iterrows():
            idx = row["index"]

            if idx == 0 and len(cur_words) > 0:
                sentences.append(cur_words)
                labels.append(cur_tags)
                cur_words = []
                cur_tags = []

            cur_words.append(str(row["word"]))
            cur_tags.append(str(row["tag"]))

        if len(cur_words) > 0:
            sentences.append(cur_words)
            labels.append(cur_tags)

    return sentences, labels

In [9]:
from torch.utils.data import Dataset, DataLoader

label2id = {
    "O": 0,
    "B-HOS": 1,
    "I-HOS": 2,
    "B-T": 1,
    "I-T": 2,
}

id2label = {
    0: "O",
    1: "B-HOS",
    2: "I-HOS",
}


class TextDataset(Dataset):
    def __init__(self, texts, tags, tokenizer, max_len):
        self.texts = texts
        self.tags = tags
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts[idx]
        tags = self.tags[idx]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids(batch_index=0)

        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                tag = tags[word_idx]

                if tag not in label2id:
                    raise ValueError(f"Unknown tag: {tag}")

                label_ids.append(label2id[tag])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }

        return item


def create_dataloader(texts, tags, batch_size, tokenizer, max_len, shuffle=True):
    dataset = TextDataset(texts, tags, tokenizer, max_len)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [10]:
batch_size = 64

train_dataloader = create_dataloader(
    *prepare_data(train_path),
    batch_size=batch_size,
    tokenizer=tokenizer,
    max_len=64
)

dev_dataloader = create_dataloader(
    *prepare_data(dev_path),
    batch_size=batch_size,
    tokenizer=tokenizer,
    max_len=64,
    shuffle=False
)

test_dataloader = create_dataloader(
    *prepare_data(test_path),
    batch_size=batch_size,
    tokenizer=tokenizer,
    max_len=64,
    shuffle=False
)

# Create Model

In [11]:
def calculate_f1(preds, y):
    max_preds = preds.argmax(dim = 1, keepdim = True) # get the index of the max probability
    return f1_score(y.cpu(), max_preds.cpu(), average='macro')

In [12]:
class MultiTaskModel(nn.Module):
    def __init__(self, input_model, num_labels=3):
        super(MultiTaskModel, self).__init__()
        self.bert = input_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        last_hidden_state = output.last_hidden_state
        last_hidden_state = self.dropout(last_hidden_state)

        logits = self.classifier(last_hidden_state)

        return logits

In [13]:
def train(
    model,
    train_dataloader,
    dev_dataloader,
    criterion_span,
    optimizer_spans,
    device,
    num_epochs
):
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        print("Epoch:", epoch + 1)

        for batch in tqdm(train_dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer_spans.zero_grad()

            logits = model(input_ids, attention_mask)

            loss = criterion_span(
                logits.view(-1, logits.shape[-1]),
                labels.view(-1)
            )

            loss.backward()
            optimizer_spans.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_dataloader)

        val_loss = 0.0
        model.eval()

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(dev_dataloader):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask)

                loss = criterion_span(
                    logits.view(-1, logits.shape[-1]),
                    labels.view(-1)
                )

                val_loss += loss.item()

                preds = torch.argmax(logits, dim=-1)

                mask = labels != -100

                all_preds.extend(preds[mask].cpu().numpy().tolist())
                all_labels.extend(labels[mask].cpu().numpy().tolist())

        val_loss = val_loss / len(dev_dataloader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        print("Training loss:", train_loss)
        print("Validation loss:", val_loss)
        print("Validation macro F1:", val_f1)

# Start Training

In [14]:
import torch.optim as optim

num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskModel(
    input_model=input_model,
    num_labels=3
)

model.to(device)

criterion_span = nn.CrossEntropyLoss(ignore_index=-100)

optimizer_spans = optim.Adam(
    model.parameters(),
    lr=2e-5,
    weight_decay=1e-5
)

train(
    model=model,
    train_dataloader=train_dataloader,
    dev_dataloader=dev_dataloader,
    criterion_span=criterion_span,
    optimizer_spans=optimizer_spans,
    device=device,
    num_epochs=num_epochs
)

Epoch: 1


100%|██████████| 18/18 [00:12<00:00,  1.48it/s]


Training loss: 0.4751724904175285
Validation loss: 0.37470855977800155
Validation macro F1: 0.6918371694904583
Epoch: 2


100%|██████████| 18/18 [00:12<00:00,  1.49it/s]


Training loss: 0.34888795897257413
Validation loss: 0.3627176599370109
Validation macro F1: 0.7070891740732218
Epoch: 3


...

Training loss: 0.10849304427560284
Validation loss: 0.4619314720233281
Validation macro F1: 0.7286139467308949
Epoch: 9


100%|██████████| 18/18 [00:12<00:00,  1.49it/s]


Training loss: 0.08893086852358399
Validation loss: 0.5200583504305946
Validation macro F1: 0.7255441777095194
Epoch: 10


100%|██████████| 18/18 [00:12<00:00,  1.50it/s]

Training loss: 0.07272120050603537
Validation loss: 0.5899165024360021
Validation macro F1: 0.7228284877955055


In [15]:
torch.save(model.state_dict(), "/kaggle/working/xlm-r-large.pt")
print("Saved to /kaggle/working/xlm-r-large.pt")

Saved to /kaggle/working/xlm-r-large.pt


In [16]:
import ast
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support


def parse_gold_spans(value):
    if pd.isna(value):
        return set()

    if isinstance(value, str):
        value = value.strip()
        if value == "" or value == "[]":
            return set()
        try:
            value = ast.literal_eval(value)
        except Exception:
            return set()

    result = set()

    def collect(x):
        if isinstance(x, int):
            result.add(x)
        elif isinstance(x, float) and not np.isnan(x):
            result.add(int(x))
        elif isinstance(x, (list, tuple, set)):
            for item in x:
                collect(item)

    collect(value)
    return result


def get_text_and_span_columns(df):
    possible_text_cols = ["content", "text", "comment", "sentence"]
    possible_span_cols = ["index_spans", "span_ids", "spans", "labels"]

    text_col = None
    span_col = None

    for col in possible_text_cols:
        if col in df.columns:
            text_col = col
            break

    for col in possible_span_cols:
        if col in df.columns:
            span_col = col
            break

    if text_col is None or span_col is None:
        raise ValueError(
            f"Không tìm được cột text/span. Columns hiện có: {df.columns.tolist()}"
        )

    return text_col, span_col


def load_bio_sentences(file_path):
    df = pd.read_csv(file_path)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df.columns = [c.strip().lower() for c in df.columns]

    if "sentence_id" in df.columns:
        sentences = []
        for _, group in df.groupby("sentence_id", sort=False):
            words = group["word"].astype(str).tolist()
            sentences.append(words)
        return sentences

    sentences = []
    cur_words = []

    for _, row in df.iterrows():
        idx = row["index"]

        if idx == 0 and len(cur_words) > 0:
            sentences.append(cur_words)
            cur_words = []

        cur_words.append(str(row["word"]))

    if len(cur_words) > 0:
        sentences.append(cur_words)

    return sentences


def align_tokens_to_text(tokens, text):
    offsets = []
    cursor = 0

    for tok in tokens:
        tok = str(tok)

        start = text.find(tok, cursor)

        # fallback nếu không tìm thấy token chính xác
        if start == -1:
            stripped = tok.strip()
            start = text.find(stripped, cursor)

        if start == -1:
            # Không align được thì đánh dấu None
            offsets.append(None)
            continue

        end = start + len(tok)
        offsets.append((start, end))
        cursor = end

    return offsets


def pred_labels_to_char_indices(tokens, pred_labels, text):
    offsets = align_tokens_to_text(tokens, text)
    pred_chars = set()

    for label, offset in zip(pred_labels, offsets):
        if offset is None:
            continue

        if label != 0:
            start, end = offset
            pred_chars.update(range(start, end))

    return pred_chars


def count_contiguous_spans(char_indices):
    if not char_indices:
        return 0

    sorted_idx = sorted(char_indices)
    count = 1

    for i in range(1, len(sorted_idx)):
        if sorted_idx[i] != sorted_idx[i - 1] + 1:
            count += 1

    return count


def prf_from_char_sets(gold_sets, pred_sets):
    tp = 0
    pred_total = 0
    gold_total = 0

    for gold, pred in zip(gold_sets, pred_sets):
        tp += len(gold & pred)
        pred_total += len(pred)
        gold_total += len(gold)

    precision = tp / pred_total if pred_total > 0 else 0.0
    recall = tp / gold_total if gold_total > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    return precision, recall, f1


def predict_bio_labels_for_sentences(model, tokenizer, sentences, device, max_len=64):
    model.eval()
    all_sentence_preds = []

    with torch.no_grad():
        for words in tqdm(sentences):
            encoding = tokenizer(
                words,
                is_split_into_words=True,
                max_length=max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

            input_ids = encoding["input_ids"].to(device)
            attention_mask = encoding["attention_mask"].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy().tolist()

            word_ids = encoding.word_ids(batch_index=0)

            token_preds = []
            previous_word_idx = None

            for pred, word_idx in zip(preds, word_ids):
                if word_idx is None:
                    continue

                # lấy prediction ở subword đầu tiên của mỗi token
                if word_idx != previous_word_idx:
                    token_preds.append(pred)

                previous_word_idx = word_idx

            # Nếu bị truncate thì pad phần còn lại là O
            if len(token_preds) < len(words):
                token_preds += [0] * (len(words) - len(token_preds))

            all_sentence_preds.append(token_preds[:len(words)])

    return all_sentence_preds


def evaluate_span_level_official(
    model,
    tokenizer,
    test_bio_path,
    test_gold_path,
    device,
    model_name="xlm-roberta-large",
    max_len=64,
    save_csv=True
):
    test_sentences = load_bio_sentences(test_bio_path)

    gold_df = pd.read_csv(test_gold_path)
    gold_df = gold_df.loc[:, ~gold_df.columns.str.contains("^Unnamed")]
    gold_df.columns = [c.strip() for c in gold_df.columns]

    text_col, span_col = get_text_and_span_columns(gold_df)

    gold_texts = gold_df[text_col].astype(str).tolist()
    gold_sets = gold_df[span_col].apply(parse_gold_spans).tolist()

    if len(test_sentences) != len(gold_texts):
        print("WARNING: Số câu BIO và số dòng gold test.csv không khớp!")
        print("BIO sentences:", len(test_sentences))
        print("Gold rows:", len(gold_texts))
        n = min(len(test_sentences), len(gold_texts))
        test_sentences = test_sentences[:n]
        gold_texts = gold_texts[:n]
        gold_sets = gold_sets[:n]

    pred_label_sentences = predict_bio_labels_for_sentences(
        model=model,
        tokenizer=tokenizer,
        sentences=test_sentences,
        device=device,
        max_len=max_len
    )

    pred_sets = []

    for tokens, pred_labels, text in zip(test_sentences, pred_label_sentences, gold_texts):
        pred_chars = pred_labels_to_char_indices(
            tokens=tokens,
            pred_labels=pred_labels,
            text=text
        )
        pred_sets.append(pred_chars)

    single_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) == 1
    ]

    multiple_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) > 1
    ]

    all_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) >= 1
    ]

    def subset_score(indices):
        subset_gold = [gold_sets[i] for i in indices]
        subset_pred = [pred_sets[i] for i in indices]
        return prf_from_char_sets(subset_gold, subset_pred)

    single_p, single_r, single_f1 = subset_score(single_indices)
    multiple_p, multiple_r, multiple_f1 = subset_score(multiple_indices)
    all_p, all_r, all_f1 = subset_score(all_indices)

    result = {
        "model": model_name,

        "single_precision": single_p,
        "single_recall": single_r,
        "single_f1": single_f1,

        "multiple_precision": multiple_p,
        "multiple_recall": multiple_r,
        "multiple_f1": multiple_f1,

        "all_precision": all_p,
        "all_recall": all_r,
        "all_f1": all_f1,

        "n_single": len(single_indices),
        "n_multiple": len(multiple_indices),
        "n_all": len(all_indices),
    }

    print("===== Span-level official-style benchmark =====")
    for k, v in result.items():
        if k == "model":
            print(f"{k}: {v}")
        elif k.startswith("n_"):
            print(f"{k}: {v}")
        else:
            print(f"{k}: {v:.6f}")

    if save_csv:
        result_path = Path("/kaggle/working/xlmr_large_span_level_results.csv")

        if result_path.exists():
            old_df = pd.read_csv(result_path)
            new_df = pd.concat([old_df, pd.DataFrame([result])], ignore_index=True)
        else:
            new_df = pd.DataFrame([result])

        new_df.to_csv(result_path, index=False)
        print(f"\nSaved result to: {result_path}")

    return result, pred_sets, gold_sets

# Load and test

In [17]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import pandas as pd
from pathlib import Path

In [18]:
def test(model, test_dataloader, device, model_name="xlm-roberta-large", save_csv=True):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(test_dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=-1)

            # Bỏ qua padding/subword
            mask = labels != -100

            all_preds.extend(preds[mask].cpu().numpy().tolist())
            all_labels.extend(labels[mask].cpu().numpy().tolist())

    accuracy = accuracy_score(all_labels, all_preds)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    result = {
        "model": model_name,
        "accuracy": accuracy,

        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,

        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,

        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

    print("===== Overall token-level BIO scores =====")
    for k, v in result.items():
        if k != "model":
            print(f"{k}: {v:.6f}")
        else:
            print(f"{k}: {v}")

    print("\n===== Per-label report =====")
    print(
        classification_report(
            all_labels,
            all_preds,
            labels=[0, 1, 2],
            target_names=["O", "B-HOS", "I-HOS"],
            zero_division=0
        )
    )

    print("\n===== Confusion matrix =====")
    print(confusion_matrix(all_labels, all_preds, labels=[0, 1, 2]))

    if save_csv:
        result_path = Path("/kaggle/working/xlmr_large_token_level_results.csv")

        if result_path.exists():
            old_df = pd.read_csv(result_path)
            new_df = pd.concat([old_df, pd.DataFrame([result])], ignore_index=True)
        else:
            new_df = pd.DataFrame([result])

        new_df.to_csv(result_path, index=False)
        print(f"\nSaved result to: {result_path}")

    return result, all_preds, all_labels

In [19]:
model = MultiTaskModel(input_model = input_model, num_labels=3)
model.load_state_dict(torch.load("/kaggle/working/xlm-r-large.pt"))
model.to(device)

span_result_base, pred_spans_base, gold_spans = evaluate_span_level_official(
    model=model,
    tokenizer=tokenizer,
    test_bio_path=test_bio_path,
    test_gold_path=test_gold_path,
    device=device,
    model_name="xlm-roberta-large",
    max_len=64,
    save_csv=True
)

100%|██████████| 1106/1106 [00:22<00:00, 49.99it/s]


===== Span-level official-style benchmark =====
model: xlm-roberta-large
single_precision: 0.745124
single_recall: 0.517148
single_f1: 0.610549
multiple_precision: 0.754711
multiple_recall: 0.625408
multiple_f1: 0.684002
all_precision: 0.752711
all_recall: 0.599503
all_f1: 0.667428
n_single: 235
n_multiple: 296
n_all: 531

Saved result to: /kaggle/working/xlmr_large_span_level_results.csv
